# IMPORTING DEPENDENCIES

In [ ]:
import os
from pathlib import Path

# Robustly locate the project root (the folder containing the dataset/ directory)
# Works regardless of which directory VS Code starts the Jupyter kernel in.
_search = Path(os.getcwd())
for _candidate in [_search] + list(_search.parents):
    if (_candidate / "dataset").exists():
        os.chdir(_candidate)
        break

print("Working directory set to:", os.getcwd())

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_curve, auc,
    confusion_matrix, ConfusionMatrixDisplay,
    r2_score, mean_absolute_error, mean_squared_error
)

# ABOUT DATASET

In [ ]:
df = pd.read_csv("dataset/filtered/student_habits_exam_performance_filtered.csv")
df.info()

# DATASET CLEANING

In [ ]:
cols_needed = [
    'study_hours_per_day',
    'sleep_hours',
    'attendance_percentage',
    'social_media_hours',
    'netflix_hours',
    'exam_score'
]

df_analysis = df[cols_needed].copy()
print(df_analysis.shape)
print(df_analysis.dtypes)

# DATA EXPLORATION

### Data Quality Check

In [ ]:
# Missing values
print(df_analysis.isnull().sum())

# Duplicates
print(f"Duplicates: {df_analysis.duplicated().sum()}")

### Distribution of Key Variables

In [ ]:
num_cols = ['study_hours_per_day', 'sleep_hours', 'attendance_percentage',
            'social_media_hours', 'netflix_hours', 'exam_score']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    sns.histplot(df_analysis[col], kde=True, ax=axes[i//3][i%3])
    axes[i//3][i%3].set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

### Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    sns.boxplot(y=df_analysis[col], ax=axes[i//3][i%3])
    axes[i//3][i%3].set_title(f'Boxplot of {col}')
plt.tight_layout()
plt.show()

### Correlation Heatmap

In [ ]:
plt.figure(figsize=(9, 6))
sns.heatmap(df_analysis[num_cols].corr(),
            annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap — Student Habits & Exam Score')
plt.tight_layout()
plt.show()

### Data Transformation

In [ ]:
# Bin social media hours into usage groups
df_analysis['sm_group'] = pd.cut(
    df_analysis['social_media_hours'],
    bins=[-float('inf'), 1, 2, 3.5, float('inf')],
    labels=['Low (<1h)', 'Moderate (1-2h)', 'High (2-3.5h)', 'Very High (>3.5h)'])

# Bin study hours
df_analysis['study_group'] = pd.cut(
    df_analysis['study_hours_per_day'],
    bins=[-float('inf'), 2, 4, 6, float('inf')],
    labels=['Low (<2h)', 'Moderate (2-4h)', 'High (4-6h)', 'Very High (>6h)'])

# High vs Low exam score (median split — used for logistic regression)
df_analysis['exam_class'] = (
    df_analysis['exam_score'] >= df_analysis['exam_score'].median()
).astype(int)

print(df_analysis['sm_group'].value_counts())
print(df_analysis['study_group'].value_counts())
print(f"Exam score median: {df_analysis['exam_score'].median():.2f}")
print(df_analysis['exam_class'].value_counts())

### EDA

In [ ]:
# Social media group vs exam score, study hours, sleep hours
print(df_analysis.groupby('sm_group', observed=True)[
    ['exam_score', 'study_hours_per_day',
     'sleep_hours', 'attendance_percentage']].mean().round(2))

print()

# Study group vs exam score
print(df_analysis.groupby('study_group', observed=True)[
    ['exam_score', 'social_media_hours', 'sleep_hours']].mean().round(2))

# LINEAR REGRESSION

### Exam Score

##### Check all predictors — identify any with p-value > 0.05 (multicollinearity)

In [ ]:
features = ['study_hours_per_day', 'sleep_hours', 'attendance_percentage',
            'social_media_hours', 'netflix_hours']

X = df_analysis[features]
y = df_analysis['exam_score']

X_const = sm.add_constant(X)
model_exam = sm.OLS(y, X_const).fit()
print(model_exam.summary())

##### All predictors significant (p < 0.001) — model is clean, no removal needed

In [ ]:
# Residual plot
y_fitted = model_exam.fittedvalues
residuals = model_exam.resid

plt.figure(figsize=(7, 5))
plt.scatter(y_fitted, residuals, alpha=0.4, color='steelblue', s=15)
plt.axhline(0, color='red', linestyle='--', linewidth=1.2)
plt.title('Linear Regression — Residuals vs Fitted')
plt.xlabel('Fitted exam_score')
plt.ylabel('Residuals')
plt.tight_layout()
plt.show()

# RANDOM FOREST

### Exam Score

In [ ]:
X_rf = df_analysis[features]
y_rf = df_analysis['exam_score']

scaler_rf = StandardScaler()
X_rf_scaled = scaler_rf.fit_transform(X_rf)

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf_scaled, y_rf, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_rf, y_train_rf)
y_pred_rf = rf.predict(X_test_rf)

r2  = r2_score(y_test_rf, y_pred_rf)
mae = mean_absolute_error(y_test_rf, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test_rf, y_pred_rf))
print(f"R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}")

In [ ]:
# Feature importances
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(7, 4))
importances.plot(kind='barh', color='steelblue')
plt.title('Random Forest — Feature Importances (exam_score)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

# LOGISTIC REGRESSION

### Exam Score (High vs Low — median split)

In [ ]:
features_log = ['study_hours_per_day', 'sleep_hours', 'attendance_percentage',
                'social_media_hours', 'netflix_hours']

X_log = df_analysis[features_log]
y_log = df_analysis['exam_class']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_log, test_size=0.2, random_state=42)

log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:, 1]

print('=== Exam Score Classifier (High vs Low) ===')
print(classification_report(y_test, y_pred, target_names=['Low', 'High']))

fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score   = auc(fpr, tpr)
print(f'AUC Score: {auc_score:.3f}')

### ROC Curve

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', linewidth=2,
         label=f'ROC Curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--',
         label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Exam Score Classifier (High vs Low)')
plt.legend()
plt.tight_layout()
plt.show()

### Confusion Matrix

In [ ]:
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Low', 'High'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Exam Score Classifier')
plt.tight_layout()
plt.show()

# STORYTELLING

#### Plot 1 — The Core Story: Social Media Usage Group vs Exam Score

In [ ]:
order = ['Low (<1h)', 'Moderate (1-2h)', 'High (2-3.5h)', 'Very High (>3.5h)']

group_data = df_analysis.groupby('sm_group', observed=True)[
    'exam_score'].mean().reindex(order)

plt.figure(figsize=(9, 5))
bars = plt.bar(order, group_data,
               color=['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'])
plt.title('Average Exam Score by Social Media Usage Group',
          fontsize=13, fontweight='bold')
plt.xlabel('Social Media Usage')
plt.ylabel('Average Exam Score')
plt.ylim(0, 100)
for bar, val in zip(bars, group_data):
    plt.text(bar.get_x() + bar.get_width()/2, val - 3,
             f'{val:.1f}', ha='center', fontweight='bold', color='white')
plt.tight_layout()
plt.show()

#### Plot 2 — Study Hours vs Exam Score (coloured by Social Media Usage)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Study hours vs exam score
scatter = axes[0].scatter(
    df_analysis['study_hours_per_day'],
    df_analysis['exam_score'],
    c=df_analysis['social_media_hours'],
    cmap='RdYlGn_r', alpha=0.4, s=15)
plt.colorbar(scatter, ax=axes[0], label='Social Media Hours')
axes[0].set_title('Study Hours vs Exam Score\n(colour = social media hours)')
axes[0].set_xlabel('Study Hours per Day')
axes[0].set_ylabel('Exam Score')

# Social media hours vs exam score
axes[1].scatter(
    df_analysis['social_media_hours'],
    df_analysis['exam_score'],
    alpha=0.3, color='#e74c3c', s=15)
axes[1].set_title('Social Media Hours vs Exam Score')
axes[1].set_xlabel('Social Media Hours per Day')
axes[1].set_ylabel('Exam Score')

plt.suptitle('More Study Time Lifts Scores; More Social Media Drags Them Down',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### Plot 3 — Sleep & Attendance Impact on Exam Score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sleep hours vs exam score
axes[0].scatter(df_analysis['sleep_hours'], df_analysis['exam_score'],
                alpha=0.3, color='#3498db', s=15)
axes[0].set_title('Sleep Hours vs Exam Score')
axes[0].set_xlabel('Sleep Hours per Night')
axes[0].set_ylabel('Exam Score')

# Attendance vs exam score
axes[1].scatter(df_analysis['attendance_percentage'], df_analysis['exam_score'],
                alpha=0.3, color='#9b59b6', s=15)
axes[1].set_title('Attendance Percentage vs Exam Score')
axes[1].set_xlabel('Attendance Percentage')
axes[1].set_ylabel('Exam Score')

plt.suptitle('Better Sleep and Attendance Associate with Higher Exam Scores',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### Plot 4 — Study Group vs All Outcomes

In [ ]:
order_s = ['Low (<2h)', 'Moderate (2-4h)', 'High (4-6h)', 'Very High (>6h)']
study_data = df_analysis.groupby('study_group', observed=True)[
    ['exam_score', 'social_media_hours', 'sleep_hours']
].mean().reindex(order_s)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_s = ['#e74c3c', '#f1c40f', '#2ecc71', '#1abc9c']

axes[0].bar(order_s, study_data['exam_score'], color=colors_s)
axes[0].set_title('Avg Exam Score by Study Group')
axes[0].set_xlabel('Study Hours Group')
axes[0].set_ylabel('Exam Score')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(order_s, study_data['social_media_hours'], color=colors_s)
axes[1].set_title('Avg Social Media Hours by Study Group')
axes[1].set_xlabel('Study Hours Group')
axes[1].set_ylabel('Social Media Hours')
axes[1].tick_params(axis='x', rotation=15)

axes[2].bar(order_s, study_data['sleep_hours'], color=colors_s)
axes[2].set_title('Avg Sleep Hours by Study Group')
axes[2].set_xlabel('Study Hours Group')
axes[2].set_ylabel('Sleep Hours')
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle('Students Who Study More Have Higher Scores and Lower Social Media Use',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### Plot 5 — Model Summary: OLS Coefficients & Random Forest Importances

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# OLS coefficients (unscaled, so comparable in effect size)
ols_features = ['study_hours_per_day', 'sleep_hours',
                 'attendance_percentage', 'social_media_hours', 'netflix_hours']
ols_coefs    = [model_exam.params[f] for f in ols_features]
colors_ols   = ['#2ecc71' if c > 0 else '#e74c3c' for c in ols_coefs]

axes[0].barh(ols_features, ols_coefs, color=colors_ols)
axes[0].axvline(x=0, color='black', linewidth=0.8)
axes[0].set_title(f'OLS Regression Coefficients\nR²={model_exam.rsquared:.3f} (full dataset)')
axes[0].set_xlabel('Coefficient (effect per 1-unit increase)')

# Random Forest importances
imp_sorted = importances.sort_values(ascending=True)
axes[1].barh(imp_sorted.index, imp_sorted.values, color='steelblue')
axes[1].set_title(f'Random Forest Feature Importances\nTest R²={r2:.3f}')
axes[1].set_xlabel('Importance score')

plt.suptitle('What Predicts Exam Score? — OLS vs Random Forest',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()